In [2]:
import pdfplumber
import re
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [3]:
import sys

sys.path.append("..")

from preprocessing.text_cleaner import advanced_clean_text

from preprocessing.skill_extractor import advanced_skill_extractor

from pdf_parser.pdf_reader import extract_text_from_pdf

from matching.semantic_matcher import calculate_semantic_similarity

from utils.explainability import (
    generate_match_explanation
)

In [4]:
pdf_path = "../sample_resume/sample.pdf"

full_text = extract_text_from_pdf(pdf_path)

print(full_text[:3000])

INFORMATION TECHNOLOGY MANAGER
Summary
Dedicated IT Manager well-versed in analyzing and mitigating risk and finding cost-effective solutions. Excels at boosting performance and
productivity by establishing realistic goals and enforcing deadlines.
Highlights
Operations management
Salary structure/compensation analysis
Project trackingÂ
Calm under pressure
Performance criteria tracking
Compensation/benefits administration
Waterfall framework
Staff development
Scrum methodology
Client communication
Enterprise platforms
Experience
Information Technology Manager , 03/2013 to Current Company Name ï¼​ City , State
Managed a four-person local IT team, allocating resources to ongoing projects and enforcing deadlines.
Drove business KPIs through rapid iteration of customer-facing product features.
Leveraged in-depth understanding of end-to-end customer experience to identify pain points and latent customer needs.
Collaborated with the global team to resolve IT support cases.
Build and maintain 

In [5]:
cleaned_pdf_resume = advanced_clean_text(
    full_text
)

print(cleaned_pdf_resume[:2000])

information technology manager summary dedicated manager well versed analyzing mitigating risk finding cost effective solution excels boosting performance productivity establishing realistic goal enforcing deadline highlight operation management salary structure compensation analysis project tracking calm pressure performance criterion tracking compensation benefit administration waterfall framework staff development scrum methodology client communication enterprise platform experience information technology manager current company name city state managed four person local team allocating resource ongoing project enforcing deadline drove business kpis rapid iteration customer facing product feature leveraged depth understanding end end customer experience identify pain point latent customer need collaborated global team resolve support case build maintain staff five terminate cause one employee create audit process interlocking team adjust required manage travel budget staff site visit

In [6]:
pdf_resume_skills = advanced_skill_extractor(
    cleaned_pdf_resume
)

print("Extracted Skills:")

print(pdf_resume_skills)

Extracted Skills:
['sql', 'communication']


In [7]:
import pandas as pd

job_df = pd.read_csv(
    "../datasets/jobs.csv"
)

job_df.head()

,Unnamed: 0,Job Title,Job Description
0,0,Flutter Developer,We are looking for hire experts flutter develo...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,4,Full Stack Developer,job responsibility full stack engineer – react...


In [8]:
job_df["cleaned_job_description"] = (
    job_df["Job Description"]
    .astype(str)
    .apply(advanced_clean_text)
)

In [9]:
sample_job = job_df[
    "cleaned_job_description"
][0]

semantic_score = calculate_semantic_similarity(
    cleaned_pdf_resume,
    sample_job
)

print("Semantic Match Score:")

print(f"{semantic_score}%")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Semantic Match Score:
38.40999984741211%


In [10]:
all_jobs = job_df[
    "cleaned_job_description"
].tolist()

job_titles = job_df[
    "Job Title"
].tolist()

In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
job_embeddings = model.encode(
    all_jobs
)

In [13]:
resume_embedding = model.encode(
    cleaned_pdf_resume
)

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

all_scores = cosine_similarity(
    [resume_embedding],
    job_embeddings
)

In [15]:
scores = all_scores[0]

In [16]:
results_df = pd.DataFrame({

    "Job Title": job_titles,

    "Match Score": scores
})

In [17]:
results_df["Match Score"] = (
    results_df["Match Score"] * 100
).round(2)

In [18]:
top_jobs = results_df.sort_values(
    by="Match Score",
    ascending=False
)

In [19]:
top_jobs.head(10)

,Job Title,Match Score
1919,Database Administrator,81.389999
1655,Network Administrator,80.610001
1951,Software Engineer,80.400002
573,Java Developer,78.970001
1511,DevOps Engineer,78.690002
947,Java Developer,78.370003
942,iOS Developer,78.260002
1271,Database Administrator,77.940002
18,Database Administrator,77.440002
1141,Node js developer,77.419998


In [20]:
sample_job = job_df[
    "cleaned_job_description"
][0]

job_skills = advanced_skill_extractor(
    sample_job
)

print(job_skills)

[]


In [21]:
from matching.ats_scorer import (
    calculate_skill_overlap,

    calculate_resume_quality,

    calculate_final_ats_score
)

In [22]:
skill_overlap_score = calculate_skill_overlap(

    pdf_resume_skills,

    job_skills
)

print(skill_overlap_score)

0


In [23]:
resume_quality_score = calculate_resume_quality(
    cleaned_pdf_resume
)

print(resume_quality_score)

80


In [24]:
final_ats_score = calculate_final_ats_score(

    semantic_score,

    skill_overlap_score,

    resume_quality_score
)

print(final_ats_score)

39.36


In [25]:
explanation = generate_match_explanation(

    pdf_resume_skills,

    job_skills,

    semantic_score,

    final_ats_score
)

In [26]:
print("Matched Skills:")

print(
    explanation["matched_skills"]
)

print("\nMissing Skills:")

print(
    explanation["missing_skills"]
)

print("\nSemantic Score:")

print(
    explanation["semantic_score"]
)

print("\nATS Score:")

print(
    explanation["ats_score"]
)

Matched Skills:
[]

Missing Skills:
[]

Semantic Score:
38.41

ATS Score:
39.36


In [27]:
import sys
sys.path.append("..")


In [28]:
import google.generativeai as genai

for model in genai.list_models():
    print(model.name)

C:\Users\Meghna Tomar\AppData\Local\Temp\ipykernel_9868\298300085.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [29]:
missing_skills = explanation["missing_skills"]

In [30]:
resume_skills = advanced_skill_extractor(
    cleaned_pdf_resume
)

best_job_title = top_jobs["Job Title"]

In [31]:
import importlib
from utils import llm_feedback

importlib.reload(llm_feedback)

<module 'utils.llm_feedback' from 'd:\\MEGHNA TOMAR\\TalentSyncAI\\notebooks\\..\\utils\\llm_feedback.py'>

In [32]:
from utils.llm_feedback import (
    generate_resume_feedback
)

In [33]:
feedback = generate_resume_feedback(

    cleaned_pdf_resume,

    resume_skills,

    missing_skills,

    best_job_title
)

In [34]:
print(feedback)

# Candidate Summary

The candidate presents as a highly experienced Information Technology professional with over 15 years of progressive experience, culminating in an IT Manager role. Their background demonstrates strong capabilities in traditional IT infrastructure, including network administration, server management, Active Directory, Exchange, VMWare, and SQL database administration. They possess a solid foundation in IT operations, project management, team leadership, risk mitigation, and client support, with a clear focus on ensuring system availability, security, and performance.

# Strengths

*   **Leadership & Management Experience:** The current "Information Technology Manager" role highlights strong leadership abilities, including managing teams, allocating resources, enforcing deadlines, driving KPIs, and overseeing project deliverables. This demonstrates valuable soft skills and organizational capabilities.
*   **Extensive IT Infrastructure Background:** Deep experience ac

In [35]:
from utils.career_roadmap import (
    generate_career_roadmap
)

In [36]:
roadmap = generate_career_roadmap(

    best_job_title,

    resume_skills,

    missing_skills
)


In [37]:
print(roadmap)

As an experienced career mentor, I understand your goal is to land a role in the tech industry, and you've provided a diverse list of target job titles. Given your current skills (`sql`, `communication`) and the need for high employability, I recommend focusing on a **Software Engineer (Full-Stack Web Developer)** path, specifically leveraging JavaScript/Node.js for backend and React for frontend. This combination is highly in-demand, versatile, and allows you to immediately apply your existing SQL knowledge.

This 30-day roadmap is intense but achievable with dedicated effort (aim for 2-4 hours daily).

---

## 30-Day Learning Roadmap: Full-Stack Web Developer (Node.js/React/SQL Focus)

**Target Outcome:** Build a foundational understanding of full-stack web development, create a small portfolio project, and be ready to learn more advanced concepts.

### Week 1: Foundations of Web Development & Version Control (Git & Frontend Basics)

**Goal:** Establish a strong base with essential t